# Regularization Methods

This notebook is the interactive companion to `notes.md`. It focuses on the most
computationally useful regularization ideas: shrinkage, weight decay, dropout,
early stopping, augmentation-style regularization, label smoothing, and spectral
normalization.

**Coverage:** ridge shrinkage, AdamW vs coupled $L_2$, soft-thresholding,
dropout expectation, stochastic depth, early stopping as spectral filtering,
mixup, label smoothing, spectral normalization, and parameter-group design for
transformer training.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

try:
    import seaborn as sns
    sns.set_theme(style="whitegrid", palette="colorblind")
    HAS_SNS = True
except ImportError:
    plt.style.use("seaborn-v0_8-whitegrid")
    HAS_SNS = False

mpl.rcParams.update({
    "figure.figsize":    (10, 6),
    "figure.dpi":         120,
    "font.size":           13,
    "axes.titlesize":      15,
    "axes.labelsize":      13,
    "xtick.labelsize":     11,
    "ytick.labelsize":     11,
    "legend.fontsize":     11,
    "legend.framealpha":   0.85,
    "lines.linewidth":      2.0,
    "axes.spines.top":     False,
    "axes.spines.right":   False,
    "savefig.bbox":       "tight",
    "savefig.dpi":         150,
})
np.random.seed(42)
print("Plot setup complete.")


In [ ]:
COLORS = {
    "primary": "#0077BB",
    "secondary": "#EE7733",
    "tertiary": "#009988",
    "error": "#CC3311",
    "neutral": "#555555",
    "highlight": "#EE3377",
}

def header(title):
    print("\n" + "=" * 78)
    print(title)
    print("=" * 78)

def check_close(name, value, target, tol=1e-8):
    ok = np.allclose(value, target, atol=tol, rtol=tol)
    print(f"{'PASS' if ok else 'FAIL'} - {name}")
    if not ok:
        print("  value :", value)
        print("  target:", target)
    return ok

def check_true(name, condition):
    print(f"{'PASS' if condition else 'FAIL'} - {name}")
    return condition

def ridge_solve(X, y, lam):
    d = X.shape[1]
    return np.linalg.solve(X.T @ X + lam * np.eye(d), X.T @ y)

def soft_threshold(z, lam):
    z = np.asarray(z)
    return np.sign(z) * np.maximum(np.abs(z) - lam, 0.0)

def power_iteration(W, steps=30):
    rng = np.random.default_rng(42)
    v = rng.normal(size=W.shape[1])
    v = v / np.linalg.norm(v)
    for _ in range(steps):
        u = W @ v
        u = u / (np.linalg.norm(u) + 1e-12)
        v = W.T @ u
        v = v / (np.linalg.norm(v) + 1e-12)
    sigma = float(u @ (W @ v))
    return sigma, u, v

def make_poly_features(x, degree):
    x = np.asarray(x).reshape(-1, 1)
    feats = [x**k for k in range(degree + 1)]
    return np.concatenate(feats, axis=1)

def poly_gd_path(Xtr, ytr, Xval, yval, steps=2000, lr=0.02):
    theta = np.zeros(Xtr.shape[1])
    train_losses = []
    val_losses = []
    thetas = []
    for _ in range(steps):
        resid = Xtr @ theta - ytr
        grad = Xtr.T @ resid / len(ytr)
        theta = theta - lr * grad
        train_losses.append(0.5 * np.mean((Xtr @ theta - ytr) ** 2))
        val_losses.append(0.5 * np.mean((Xval @ theta - yval) ** 2))
        thetas.append(theta.copy())
    return np.array(thetas), np.array(train_losses), np.array(val_losses)

def softmax(logits):
    logits = np.asarray(logits, dtype=float)
    shifted = logits - logits.max()
    expv = np.exp(shifted)
    return expv / expv.sum()

def consistency_penalty(model, xs, delta):
    out_a = model(xs)
    out_b = model(xs + delta)
    return np.mean((out_a - out_b) ** 2)

print("Helper functions ready.")


## 1. L2 regularization as shrinkage

We start with an ill-conditioned linear regression problem so we can see how
ridge regularization damps unstable directions and shrinks parameter norms.


In [ ]:
header("Ridge shrinkage on an ill-conditioned design")
rng = np.random.default_rng(42)
n, d = 60, 12
U, _ = np.linalg.qr(rng.normal(size=(n, d)))
V, _ = np.linalg.qr(rng.normal(size=(d, d)))
singular_values = np.geomspace(30.0, 0.2, d)
X = U @ np.diag(singular_values) @ V.T
theta_star = np.linspace(2.0, -1.0, d)
y = X @ theta_star + 0.40 * rng.normal(size=n)

Xtr, Xval = X[:40], X[40:]
ytr, yval = y[:40], y[40:]
lambdas = np.array([0.0, 0.1, 1.0, 10.0, 100.0])
sols = np.array([ridge_solve(Xtr, ytr, lam) for lam in lambdas])
train_mse = np.array([np.mean((Xtr @ s - ytr) ** 2) for s in sols])
val_mse = np.array([np.mean((Xval @ s - yval) ** 2) for s in sols])
norms = np.linalg.norm(sols, axis=1)

print("lambdas  :", lambdas)
print("norms    :", np.round(norms, 4))
print("train mse:", np.round(train_mse, 4))
print("val mse  :", np.round(val_mse, 4))
check_true("Parameter norms shrink as lambda grows", np.all(np.diff(norms) <= 1e-8))


In [ ]:
header("Coefficient paths under ridge regularization")
fig, ax = plt.subplots()
for j in range(d):
    ax.plot(lambdas + 1e-6, sols[:, j], marker="o", alpha=0.9)
ax.set_xscale("log")
ax.set_title("Ridge coefficient paths")
ax.set_xlabel("Regularization strength $\\lambda$")
ax.set_ylabel("Coefficient value")
fig.tight_layout()
plt.show()
check_true("Largest lambda gives the smallest coefficient norm", np.linalg.norm(sols[-1]) <= np.linalg.norm(sols[0]) + 1e-8)


**What to notice:** ridge does not usually create exact zeros. Its main bias is
smooth shrinkage, especially on ill-conditioned directions.


## 2. Weight decay vs coupled $L_2$

For vanilla SGD, the two views coincide. For adaptive optimizers, they do not.


In [ ]:
header("SGD: coupled L2 equals multiplicative weight decay")
theta = np.array([1.5, -2.0, 0.5])
grad = np.array([0.2, -0.4, 0.1])
eta = 0.05
lam = 0.10

coupled = theta - eta * (grad + lam * theta)
decayed = (1.0 - eta * lam) * theta - eta * grad
print("Coupled update :", coupled)
print("Decay update   :", decayed)
check_close("SGD forms match exactly", coupled, decayed)


In [ ]:
header("Adam-style coupling differs from AdamW")
theta = np.array([2.0, 0.20, -1.0])
grad = np.array([0.10, -0.80, 0.05])
lr = 0.10
lam = 0.20
eps = 1e-8

g_coupled = grad + lam * theta
precond_coupled = g_coupled / (np.sqrt(g_coupled**2) + eps)
theta_coupled = theta - lr * precond_coupled

precond_adamw = grad / (np.sqrt(grad**2) + eps)
theta_adamw = theta - lr * precond_adamw - lr * lam * theta

print("Adam with coupled L2 :", theta_coupled)
print("AdamW decoupled      :", theta_adamw)
check_true("Adaptive coupling changes the update", np.linalg.norm(theta_coupled - theta_adamw) > 1e-6)


In [ ]:
header("Coordinate-wise update magnitudes")
coupled_update = theta_coupled - theta
adamw_update = theta_adamw - theta
idx = np.arange(len(theta))

fig, ax = plt.subplots()
ax.bar(idx - 0.18, np.abs(coupled_update), width=0.36, label="Adam + coupled $L_2$", color=COLORS["secondary"])
ax.bar(idx + 0.18, np.abs(adamw_update), width=0.36, label="AdamW", color=COLORS["primary"])
ax.set_title("Coupled and decoupled regularization change adaptive updates")
ax.set_xlabel("Coordinate")
ax.set_ylabel("Absolute update size")
ax.legend()
fig.tight_layout()
plt.show()
check_true("At least one coordinate changes noticeably", np.max(np.abs(coupled_update - adamw_update)) > 0.02)


**Takeaway:** for adaptive optimizers, "I added an $L_2$ term" and "I used
weight decay" are not synonyms.


## 3. L1 regularization and soft thresholding

The scalar proximal problem shows exactly why $L_1$ can create zeros while
$L_2$ usually cannot.


In [ ]:
header("Soft-thresholding values")
z = np.array([-2.0, -0.6, -0.2, 0.0, 0.3, 0.9, 2.5])
lam = 0.5
prox = soft_threshold(z, lam)
print("z     :", z)
print("prox  :", prox)
check_true("Small magnitudes are driven to zero", np.count_nonzero(prox == 0.0) >= 3)


In [ ]:
header("Soft-thresholding curve")
grid = np.linspace(-2.5, 2.5, 400)
fig, ax = plt.subplots()
ax.plot(grid, grid, linestyle="--", color=COLORS["neutral"], label="Identity")
ax.plot(grid, soft_threshold(grid, lam), color=COLORS["primary"], label="Soft threshold")
ax.set_title("The proximal map of $\\lambda |\\theta|$")
ax.set_xlabel("Input $z$")
ax.set_ylabel("Output")
ax.legend()
fig.tight_layout()
plt.show()
check_true("The map is odd-symmetric", np.max(np.abs(soft_threshold(grid, lam) + soft_threshold(-grid, lam))) < 1e-10)


In [ ]:
header("Ridge vs lasso vs elastic net on correlated features")
rng = np.random.default_rng(7)
n = 80
base = rng.normal(size=n)
X = np.column_stack([
    base + 0.05 * rng.normal(size=n),
    base + 0.05 * rng.normal(size=n),
    rng.normal(size=n),
    rng.normal(size=n),
])
y = 2.5 * X[:, 0] + 0.05 * rng.normal(size=n)

ridge = ridge_solve(X, y, lam=1.0)

def lasso_cd(X, y, lam=0.25, steps=200):
    theta = np.zeros(X.shape[1])
    col_norms = np.sum(X**2, axis=0)
    for _ in range(steps):
        for j in range(X.shape[1]):
            r_j = y - X @ theta + X[:, j] * theta[j]
            rho = X[:, j] @ r_j
            theta[j] = soft_threshold(rho / col_norms[j], lam / col_norms[j])
    return theta

lasso = lasso_cd(X, y, lam=0.20)

def elastic_net_cd(X, y, lam1=0.15, lam2=0.30, steps=200):
    theta = np.zeros(X.shape[1])
    col_norms = np.sum(X**2, axis=0)
    for _ in range(steps):
        for j in range(X.shape[1]):
            r_j = y - X @ theta + X[:, j] * theta[j]
            rho = X[:, j] @ r_j
            theta[j] = soft_threshold(rho, lam1) / (col_norms[j] + lam2)
    return theta

elastic = elastic_net_cd(X, y)
print("Ridge      :", np.round(ridge, 3))
print("Lasso      :", np.round(lasso, 3))
print("Elastic net:", np.round(elastic, 3))
check_true("Lasso yields at least one exact zero", np.any(np.isclose(lasso, 0.0)))


**What to notice:** $L_1$ induces exact zeros, while elastic net blends sparsity
with the stabilizing effect of quadratic shrinkage.


## 4. Dropout as multiplicative noise

With inverted dropout, the mean activation is preserved but the variance grows
as the keep probability falls.


In [ ]:
header("Inverted dropout preserves expectation")
rng = np.random.default_rng(42)
h = np.array([1.5, -2.0, 0.5, 3.0])
q = 0.7
samples = []
for _ in range(30000):
    mask = (rng.random(h.shape) < q).astype(float)
    samples.append(mask * h / q)
samples = np.array(samples)
empirical_mean = samples.mean(axis=0)
print("Original activations :", h)
print("Empirical mean       :", np.round(empirical_mean, 3))
check_close("Expectation is preserved", empirical_mean, h, tol=0.05)


In [ ]:
header("Dropout variance grows as keep probability shrinks")
empirical_var = samples.var(axis=0)
theoretical_var = h**2 * (1.0 - q) / q
print("Empirical variance  :", np.round(empirical_var, 3))
print("Theoretical variance:", np.round(theoretical_var, 3))
check_close("Variance formula matches simulation", empirical_var, theoretical_var, tol=0.12)


In [ ]:
header("One-coordinate dropout distribution")
fig, ax = plt.subplots()
ax.hist(samples[:, 0], bins=30, color=COLORS["primary"], alpha=0.85)
ax.axvline(h[0], color=COLORS["error"], linestyle="--", label="Original value")
ax.set_title("Distribution of a dropped coordinate under inverted dropout")
ax.set_xlabel("Sampled activation")
ax.set_ylabel("Count")
ax.legend()
fig.tight_layout()
plt.show()
check_true("Dropped coordinate occasionally becomes zero", np.any(np.isclose(samples[:, 0], 0.0)))


## 5. Stochastic depth

Residual-path masking is a layerwise version of dropout used heavily in deep
vision architectures.


In [ ]:
header("Expected active depth under stochastic depth")
rng = np.random.default_rng(12)
L = 48
survival = np.linspace(1.0, 0.6, L)
runs = []
for _ in range(5000):
    masks = rng.random(L) < survival
    runs.append(masks.sum())
runs = np.array(runs)
print("Average active blocks :", runs.mean())
print("Expected active blocks:", survival.sum())
check_close("Simulation matches expectation", runs.mean(), survival.sum(), tol=0.35)


## 6. Early stopping as path regularization

We will fit a high-degree polynomial by gradient descent and watch the validation
loss identify a better stopping time than the final iterate.


In [ ]:
header("Polynomial regression path for early stopping")
rng = np.random.default_rng(1)
x = np.linspace(-1.0, 1.0, 90)
y_true = np.sin(2.5 * np.pi * x)
y = y_true + 0.18 * rng.normal(size=len(x))

xtr, xval, xtest = x[:45], x[45:70], x[70:]
ytr, yval, ytest = y[:45], y[45:70], y[70:]
Xtr = make_poly_features(xtr, degree=18)
Xval = make_poly_features(xval, degree=18)
Xtest = make_poly_features(xtest, degree=18)

scale = Xtr.std(axis=0, keepdims=True)
scale[:, 0] = 1.0
Xtr = Xtr / scale
Xval = Xval / scale
Xtest = Xtest / scale

thetas, train_losses, val_losses = poly_gd_path(Xtr, ytr, Xval, yval, steps=2500, lr=0.02)
best_idx = int(np.argmin(val_losses))
print("Best validation step:", best_idx)
print("Best validation loss:", float(val_losses[best_idx]))
print("Final validation loss:", float(val_losses[-1]))
check_true("Early stopping beats the final iterate on validation", val_losses[best_idx] <= val_losses[-1] + 1e-12)


In [ ]:
header("Training and validation curves")
fig, ax = plt.subplots()
ax.plot(train_losses, color=COLORS["primary"], label="Train loss")
ax.plot(val_losses, color=COLORS["secondary"], label="Validation loss")
ax.axvline(best_idx, color=COLORS["highlight"], linestyle="--", label="Early stop")
ax.set_title("Early stopping chooses an iterate, not a new objective")
ax.set_xlabel("Gradient steps")
ax.set_ylabel("Half MSE")
ax.legend()
fig.tight_layout()
plt.show()
check_true("Validation minimum occurs before the final iterate", best_idx < len(val_losses) - 1)


In [ ]:
header("Test loss: early stop vs final iterate")
theta_early = thetas[best_idx]
theta_final = thetas[-1]
test_early = 0.5 * np.mean((Xtest @ theta_early - ytest) ** 2)
test_final = 0.5 * np.mean((Xtest @ theta_final - ytest) ** 2)
print("Early-stop test loss:", float(test_early))
print("Final-step test loss:", float(test_final))
check_true("Early stopping improves held-out performance here", test_early <= test_final + 1e-12)


## 7. Spectral-filter view of early stopping

For least squares, both ridge and early stopping act like frequency-selective
filters. Here we compare their fit fractions across singular directions.


In [ ]:
header("Ridge fit fractions vs early-stopping fit fractions")
sigmas = np.geomspace(0.1, 15.0, 200)
t = 150
lr = 0.002
lam = 2.0

ridge_fraction = sigmas**2 / (sigmas**2 + lam)
early_fraction = 1.0 - (1.0 - lr * sigmas**2) ** t

fig, ax = plt.subplots()
ax.plot(sigmas, ridge_fraction, color=COLORS["primary"], label="Ridge fit fraction")
ax.plot(sigmas, early_fraction, color=COLORS["secondary"], label="Early-stop fit fraction", linestyle="--")
ax.set_xscale("log")
ax.set_title("Both methods damp fragile singular directions")
ax.set_xlabel("Singular value $\\sigma_i$")
ax.set_ylabel("Fit fraction")
ax.legend()
fig.tight_layout()
plt.show()
check_true("Small singular values are strongly damped by both filters", ridge_fraction[0] < ridge_fraction[-1] and early_fraction[0] < early_fraction[-1])


## 8. Noise injection

A simple input-noise augmentation can make a model less brittle to perturbed test
inputs.


In [ ]:
header("Training with small input noise can improve perturbed-test robustness")
rng = np.random.default_rng(3)
xtr = np.linspace(-1.0, 1.0, 40)
ytr = np.sin(np.pi * xtr) + 0.05 * rng.normal(size=len(xtr))
Xtr = make_poly_features(xtr, degree=10)
theta_clean = ridge_solve(Xtr, ytr, lam=1e-3)

x_noisy = np.repeat(xtr, 5) + 0.05 * rng.normal(size=5 * len(xtr))
y_noisy = np.repeat(ytr, 5)
Xnoisy = make_poly_features(x_noisy, degree=10)
theta_noisy = ridge_solve(Xnoisy, y_noisy, lam=1e-3)

xt = np.linspace(-1.0, 1.0, 120)
yt = np.sin(np.pi * xt)
xpert = xt + 0.04 * rng.normal(size=len(xt))
clean_err = np.mean((make_poly_features(xpert, 10) @ theta_clean - yt) ** 2)
noisy_err = np.mean((make_poly_features(xpert, 10) @ theta_noisy - yt) ** 2)
print("Perturbed-test MSE without noise augmentation:", float(clean_err))
print("Perturbed-test MSE with noise augmentation   :", float(noisy_err))
check_true("Noise-augmented fit is at least as robust here", noisy_err <= clean_err + 0.02)


## 9. Data augmentation as vicinal risk minimization

We will create a simple geometric augmentation on a toy 2D classification dataset.


In [ ]:
header("Generate original and augmented 2D samples")
rng = np.random.default_rng(9)
class_a = rng.normal(loc=(-1.0, 0.5), scale=(0.20, 0.25), size=(30, 2))
class_b = rng.normal(loc=(1.0, -0.4), scale=(0.25, 0.20), size=(30, 2))
original = np.vstack([class_a, class_b])
labels = np.array([0] * len(class_a) + [1] * len(class_b))

jitter = rng.normal(scale=0.10, size=original.shape)
augmented = original + jitter
print("Original shape :", original.shape)
print("Augmented shape:", augmented.shape)
check_true("Augmentation keeps sample count fixed here", augmented.shape == original.shape)


In [ ]:
header("Original vs augmented data cloud")
fig, ax = plt.subplots()
ax.scatter(original[labels == 0, 0], original[labels == 0, 1], color=COLORS["primary"], label="Class 0", alpha=0.75)
ax.scatter(original[labels == 1, 0], original[labels == 1, 1], color=COLORS["secondary"], label="Class 1", alpha=0.75)
ax.scatter(augmented[labels == 0, 0], augmented[labels == 0, 1], color=COLORS["primary"], alpha=0.20)
ax.scatter(augmented[labels == 1, 0], augmented[labels == 1, 1], color=COLORS["secondary"], alpha=0.20)
ax.set_title("Augmentation creates a neighborhood around the empirical data")
ax.set_xlabel("$x_1$")
ax.set_ylabel("$x_2$")
ax.legend()
fig.tight_layout()
plt.show()
check_true("Jitter creates nontrivial vicinal points", np.mean(np.linalg.norm(augmented - original, axis=1)) > 0.05)


## 10. Mixup

Mixup regularizes through both the input and the target.


In [ ]:
header("Construct mixup examples")
rng = np.random.default_rng(10)
pair_a = class_a[:6]
pair_b = class_b[:6]
lam_mix = rng.beta(0.4, 0.4, size=len(pair_a))
mix_x = lam_mix[:, None] * pair_a + (1.0 - lam_mix)[:, None] * pair_b
mix_y = np.column_stack([lam_mix, 1.0 - lam_mix])

print("First three mixup lambdas:", np.round(lam_mix[:3], 3))
print("First mixed target rows:\n", np.round(mix_y[:3], 3))
check_true("Mixed labels remain valid distributions", np.allclose(mix_y.sum(axis=1), 1.0))


In [ ]:
header("Mixup points interpolate between classes")
fig, ax = plt.subplots()
ax.scatter(class_a[:, 0], class_a[:, 1], color=COLORS["primary"], label="Class 0")
ax.scatter(class_b[:, 0], class_b[:, 1], color=COLORS["secondary"], label="Class 1")
ax.scatter(mix_x[:, 0], mix_x[:, 1], color=COLORS["highlight"], label="Mixup", s=55)
for a, b, m in zip(pair_a, pair_b, mix_x):
    ax.plot([a[0], b[0]], [a[1], b[1]], color=COLORS["neutral"], alpha=0.15)
ax.set_title("Mixup moves supervision into the space between examples")
ax.set_xlabel("$x_1$")
ax.set_ylabel("$x_2$")
ax.legend()
fig.tight_layout()
plt.show()
check_true("Mixup samples lie between paired examples", np.all((mix_x[:, 0] >= np.minimum(pair_a[:, 0], pair_b[:, 0]) - 1e-9) & (mix_x[:, 0] <= np.maximum(pair_a[:, 0], pair_b[:, 0]) + 1e-9)))


## 11. Label smoothing

Smoothed targets reduce the gradient push toward infinite confidence.


In [ ]:
header("One-hot vs smoothed targets")
logits = np.array([4.5, 1.0, -0.5])
probs = softmax(logits)
y_onehot = np.array([1.0, 0.0, 0.0])
eps = 0.1
K = 3
y_smooth = (1.0 - eps) * y_onehot + (eps / K) * np.ones(K)

ce_onehot = -np.sum(y_onehot * np.log(probs))
ce_smooth = -np.sum(y_smooth * np.log(probs))
grad_onehot = probs - y_onehot
grad_smooth = probs - y_smooth

print("Probabilities     :", np.round(probs, 4))
print("Smoothed target   :", np.round(y_smooth, 4))
print("CE one-hot        :", float(ce_onehot))
print("CE smoothed       :", float(ce_smooth))
print("Gradient one-hot  :", np.round(grad_onehot, 4))
print("Gradient smoothed :", np.round(grad_smooth, 4))
check_true("Smoothed targets still sum to one", abs(y_smooth.sum() - 1.0) < 1e-12)


In [ ]:
header("Target and gradient comparison")
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
idx = np.arange(K)
axes[0].bar(idx - 0.18, y_onehot, width=0.36, color=COLORS["primary"], label="One-hot")
axes[0].bar(idx + 0.18, y_smooth, width=0.36, color=COLORS["secondary"], label="Smoothed")
axes[0].set_title("Targets")
axes[0].set_xlabel("Class")
axes[0].set_ylabel("Target mass")
axes[0].legend()

axes[1].bar(idx - 0.18, grad_onehot, width=0.36, color=COLORS["primary"], label="One-hot grad")
axes[1].bar(idx + 0.18, grad_smooth, width=0.36, color=COLORS["secondary"], label="Smoothed grad")
axes[1].set_title("Cross-entropy gradients in logit space")
axes[1].set_xlabel("Class")
axes[1].set_ylabel("$p_k - y_k$")
axes[1].legend()
fig.tight_layout()
plt.show()
check_true("Smoothing reduces target-class gradient magnitude", abs(grad_smooth[0]) < abs(grad_onehot[0]))


## 12. Spectral normalization

Spectral normalization controls worst-case linear amplification through the top
singular value.


In [ ]:
header("Power iteration estimates the spectral norm")
rng = np.random.default_rng(13)
W = rng.normal(size=(6, 4))
sigma_hat, u_hat, v_hat = power_iteration(W, steps=60)
sigma_exact = np.linalg.svd(W, compute_uv=False)[0]
print("Estimated sigma_max:", float(sigma_hat))
print("Exact sigma_max    :", float(sigma_exact))
check_close("Power iteration is accurate", sigma_hat, sigma_exact, tol=1e-5)


In [ ]:
header("Normalize the matrix by its top singular value")
W_sn = W / sigma_hat
sigma_sn = np.linalg.svd(W_sn, compute_uv=False)[0]
print("Spectral norm after normalization:", float(sigma_sn))
check_close("Normalized matrix has spectral norm 1", sigma_sn, 1.0, tol=1e-5)


In [ ]:
header("Singular values before and after normalization")
s_before = np.linalg.svd(W, compute_uv=False)
s_after = np.linalg.svd(W_sn, compute_uv=False)
idx = np.arange(len(s_before))
fig, ax = plt.subplots()
ax.plot(idx, s_before, marker="o", color=COLORS["secondary"], label="Before")
ax.plot(idx, s_after, marker="o", color=COLORS["primary"], label="After")
ax.set_title("Spectral normalization rescales the entire singular-value profile")
ax.set_xlabel("Index")
ax.set_ylabel("Singular value")
ax.legend()
fig.tight_layout()
plt.show()
check_true("Top singular value decreases after normalization", s_after[0] <= s_before[0] + 1e-12)


## 13. Function-space vs parameter-space regularization

Two parameter matrices can have the same Frobenius norm but different worst-case
amplification.


In [ ]:
header("Equal Frobenius norm, different spectral norm")
A = np.array([[2.0, 0.0], [0.0, 0.0]])
B = np.sqrt(2.0) * np.eye(2)
fro_A = np.linalg.norm(A, ord="fro")
fro_B = np.linalg.norm(B, ord="fro")
spec_A = np.linalg.svd(A, compute_uv=False)[0]
spec_B = np.linalg.svd(B, compute_uv=False)[0]
print("Frobenius norms:", fro_A, fro_B)
print("Spectral norms :", spec_A, spec_B)
check_close("Frobenius norms match", fro_A, fro_B)
check_true("Spectral norms differ", abs(spec_A - spec_B) > 0.4)


In [ ]:
header("Consistency penalty favors smoother functions")
xs = np.linspace(-2.0, 2.0, 400)
delta = 0.05

smooth_model = lambda z: np.tanh(0.8 * z)
sharp_model = lambda z: np.tanh(4.0 * z)

smooth_cons = consistency_penalty(smooth_model, xs, delta)
sharp_cons = consistency_penalty(sharp_model, xs, delta)
print("Smooth-model consistency penalty:", float(smooth_cons))
print("Sharp-model consistency penalty :", float(sharp_cons))
check_true("Smoother function changes less under perturbation", smooth_cons < sharp_cons)


## 14. Transformer and LLM parameter groups

Modern AdamW recipes do not decay every parameter identically.


In [ ]:
header("Example parameter grouping for AdamW")
fake_named_params = [
    ("tok_embeddings.weight", (32000, 4096)),
    ("layers.0.attn.q_proj.weight", (4096, 4096)),
    ("layers.0.attn.q_proj.bias", (4096,)),
    ("layers.0.mlp.up_proj.weight", (11008, 4096)),
    ("layers.0.input_layernorm.weight", (4096,)),
    ("lm_head.weight", (32000, 4096)),
]

decay, no_decay = [], []
for name, shape in fake_named_params:
    if name.endswith(".bias") or "norm" in name or len(shape) == 1:
        no_decay.append(name)
    else:
        decay.append(name)

print("Decay group:")
for name in decay:
    print("  -", name)
print("No-decay group:")
for name in no_decay:
    print("  -", name)
check_true("Bias terms are excluded from decay", any(name.endswith(".bias") for name in no_decay))
check_true("Normalization parameters are excluded from decay", any("norm" in name for name in no_decay))


## 15. Compact recap

We end with a short checklist tying the notebook's examples back to the section.


In [ ]:
header("Regularization checklist")
summary = {
    "Ridge": "shrinks unstable directions without creating exact zeros",
    "AdamW": "decouples shrinkage from adaptive gradient normalization",
    "L1": "creates zeros through a soft-thresholding bias",
    "Dropout": "preserves mean with inverted scaling but increases variance",
    "Early stopping": "regularizes by selecting an iterate on the path",
    "Mixup": "regularizes through both inputs and targets",
    "Spectral normalization": "controls worst-case linear amplification",
}
for k, v in summary.items():
    print(f"{k:24s} -> {v}")
check_true("Summary covers seven distinct mechanisms", len(summary) == 7)


The companion exercise notebook now turns these ideas into derivations, checks,
and implementation practice.


In [ ]:
header("Final verification")
check_true("Notebook executed through the final cell", True)
print("PASS - theory notebook ready for section-level validation")
